# 神经网络「叠加」入门:从玩具模型看 Anthropic 的可解释性研究

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/toy_superposition_tutorial.ipynb)

本 notebook 用 PyTorch **从零复现** Anthropic 经典论文 [*Toy Models of Superposition*](https://transformer-circuits.pub/2022/toy_model/index.html)(Elhage et al., 2022)的核心实验,带你理解一个反直觉的事实:

> 🧠 **神经网络可以在 d 个神经元里塞进 n 个特征(n > d),只要这些特征足够稀疏。**

这就是「**叠加**」(superposition),也是为什么大语言模型能在有限维度里编码海量概念。
理解它,就理解了 Anthropic 可解释性研究的起点——他们后续的 *Scaling Monosemanticity*(用 Sparse Autoencoder 拆开 Claude 模型激活)正是建立在这个基础上。

## 写给谁看

- 知道神经网络基本概念(线性层、ReLU、loss)的读者
- 想了解机器学习「可解释性」(interpretability)是怎么做的同学
- 对「Claude 内部是怎么工作的」好奇的人

**完全不需要论文背景**——本 notebook 会**从直觉**讲起,所有实验都自己手写训练。

## 你将复现的核心结论

1. **稠密**(每个特征几乎总是存在)→ 网络只学 `d` 个最重要的特征,其余直接放弃;
2. **中等稀疏** → 网络开始**叠加**特征,出现多义性(一个神经元承载多个特征);
3. **极度稀疏** → **几何相变**:特征向量自动排成正多边形(三角形、五边形…),
   把 n 个特征塞进 2 维空间,且能保持几乎可逆;
4. **加分**:训一个 **Sparse Autoencoder**,把叠加在一起的特征**重新拆开**——
   这正是 Anthropic 用来「读懂」Claude 内部的工具。

## 目录
0. 阅读须知
1. 「特征」是什么?「多义性」又是什么?
2. 玩具模型的设置
3. 实验 1:稠密数据下网络放弃了什么
4. 实验 2:稀疏数据让叠加发生
5. 实验 3:几何相变(三角形 → 五边形 → 多边形)
6. 加分:用 Sparse Autoencoder 拆开叠加的特征
7. 总结与进一步阅读


## 0. 阅读须知

> 📘 **小知识:本 notebook 中「特征」一词的含义**
> 在可解释性研究里,「特征」(feature)= **数据里有意义的一个维度或概念**。
> 比如图片的「猫耳朵」、文本的「礼貌语气」、用户行为的「上次登录」。
> 神经网络要做的事,就是**把输入里的特征识别出来,在某些神经元上点亮信号**。

> 📘 **小知识:为什么这件事重要?**
> 大模型有上千亿参数,我们没法肉眼看懂它们在算什么。
> 但如果**每个神经元正好对应一个明确的概念**,就能直接读出模型的「想法」——
> 那是可解释性的圣杯。可惜真实大模型完全不是这样,这就是「叠加」要解决的核心问题。

依赖(Kaggle 默认环境全部内置):`numpy`, `torch`, `matplotlib`。


## 1. 「特征」是什么?「多义性」又是什么?

我们做一个最朴素的设定:

- 数据里有 **n 个独立的特征**(可以想成「猫耳朵」「红色」「圆形」「汪汪叫」「会飞」5 个二元概念);
- 每个特征出现的概率是 `density`(稀疏度的反义词)。`density=1` 意味着每张图片**所有 5 个概念都同时出现**(不太现实);`density=0.1` 意味着每张图片**平均只触发 0.5 个概念**(更接近真实数据);
- 神经网络有一个**瓶颈层**只有 `d` 个神经元(d < n)——所以它必须**学会取舍**。

如果每个神经元都只代表 1 个概念,我们说神经元是「**单义的**」(monosemantic);
如果一个神经元在「猫耳朵」和「红色」上都点亮,我们说它是「**多义的**」(polysemantic)。

**论文要回答的核心问题**:在什么条件下,神经网络会选择「多义」?它能这样塞下多少特征?

让我们用代码亲手验证。


## 2. 玩具模型的设置

我们用论文中最经典的 ReLU 自编码器(autoencoder):

```text
  输入 x ∈ R^n  →  隐藏 h = W x ∈ R^d  →  输出 x̂ = ReLU(W^T h + b) ∈ R^n
```

- **同一个矩阵 W** 同时做编码(`x → h`)与解码(`h → x̂`),这是「权重共享」。论文这样设计是为了**强制网络在 d 维隐藏空间里安排好 n 个特征**;
- **目标**:让 `x̂ ≈ x`(自编码,自己重构自己);
- **损失**:加权 MSE,**重要的特征我们更在乎重构得准**:`L = sum_i I_i · (x_i - x̂_i)²`,其中 `I_i` 是特征 i 的「重要性」。

> 📘 **小知识:为什么用 ReLU 重构 + 权重共享?**
> ReLU 提供了「叠加」可行的关键非线性——它能在干扰项很小时把噪声**剪掉**。
> 权重共享让我们可以**直接通过观察 W** 来看网络学到了什么:
> W 的第 i 行就是「特征 i 在 d 维隐藏空间里的方向」。


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

torch.manual_seed(0)
np.random.seed(0)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}, device: {DEVICE}')


class ToyModel(nn.Module):
    """x → W → h(d-dim) → W^T → +b → ReLU → x̂"""
    def __init__(self, n_features: int, n_hidden: int):
        super().__init__()
        # W shape: (n_features, n_hidden); each ROW is the direction of a feature
        self.W = nn.Parameter(torch.empty(n_features, n_hidden))
        nn.init.xavier_uniform_(self.W)
        self.b = nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        h = x @ self.W                      # (batch, n_hidden)
        out = h @ self.W.T + self.b         # (batch, n_features)
        return F.relu(out)


def generate_batch(batch_size: int, n_features: int, density: float, device=DEVICE):
    """Each feature value ~ Uniform[0,1], present with probability `density`."""
    x = torch.rand(batch_size, n_features, device=device)
    mask = (torch.rand(batch_size, n_features, device=device) < density).float()
    return x * mask


def train(model, n_features, density, importance, n_steps=10_000, batch_size=1024, lr=1e-3):
    """Standard Adam loop with weighted MSE."""
    model = model.to(DEVICE)
    importance = importance.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for step in range(n_steps):
        x = generate_batch(batch_size, n_features, density)
        x_hat = model(x)
        loss = (importance * (x - x_hat) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % (n_steps // 10) == 0:
            losses.append(loss.item())
    return losses


print('ToyModel + train loop ready')


### 2.1 可视化辅助函数

我们要反复观察 W 行向量在 2D 隐藏空间里的几何分布,先准备一个画图函数。


In [ ]:
def plot_W(W, importance, title='', ax=None):
    """Plot each row of W (a feature direction) as a vector from origin.
    Color encodes importance."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(4.5, 4.5))
    W = W.detach().cpu().numpy()
    importance = importance.cpu().numpy() if hasattr(importance, 'cpu') else importance

    # Unit circle for reference
    ax.add_patch(Circle((0, 0), 1.0, fill=False, color='gray', linestyle=':', alpha=0.4))
    ax.axhline(0, color='gray', alpha=0.2)
    ax.axvline(0, color='gray', alpha=0.2)

    cmap = plt.cm.viridis
    norm = plt.Normalize(vmin=importance.min(), vmax=importance.max())
    for i, (x, y) in enumerate(W):
        c = cmap(norm(importance[i]))
        ax.arrow(0, 0, x, y, head_width=0.04, head_length=0.06,
                 fc=c, ec=c, length_includes_head=True, lw=2)
        ax.text(x * 1.15, y * 1.15, f'{i}', color=c, fontsize=10, ha='center', va='center')

    lim = 1.3
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)
    return ax


def plot_W_norms(W, ax=None, title=''):
    """Bar chart of ‖W_i‖, the 'how visible is feature i' metric."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))
    W = W.detach().cpu().numpy()
    norms = np.linalg.norm(W, axis=1)
    ax.bar(range(len(norms)), norms, color='steelblue')
    ax.set_xlabel('Feature index')
    ax.set_ylabel('‖W_i‖  (representation magnitude)')
    ax.set_title(title)
    ax.set_ylim(0, max(1.05, norms.max() * 1.1))


## 3. 实验 1:稠密数据下网络放弃了什么

**设置**:
- 5 个特征,2 个隐藏维度
- `density = 1.0`(每个特征都同时存在,极端稠密)
- 5 个特征的重要性递减:`I = [1.0, 0.7, 0.5, 0.35, 0.25]`

> 📘 **直觉预测**:5 个特征要塞进 2 维,但每次都同时出现——它们必然互相干扰。
> 网络的最优解大概率是:**只重构最重要的 2 个,其余完全放弃**。


In [ ]:
n_features = 5
n_hidden = 2
importance = torch.tensor([1.0, 0.7, 0.5, 0.35, 0.25])

model_dense = ToyModel(n_features, n_hidden)
losses = train(model_dense, n_features, density=1.0,
               importance=importance, n_steps=8_000)
print(f'最终 loss: {losses[-1]:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_W(model_dense.W, importance,
       title='Dense (density=1.0): only top-2 features survive', ax=axes[0])
plot_W_norms(model_dense.W,
             title='Feature representation magnitude', ax=axes[1])
plt.tight_layout()
plt.show()


**观察**:
- 左图:**只有 2 个特征(0 和 1)有显著向量长度**,其余几乎归零;
- 右图:`‖W_i‖` 在前 2 个上接近 1,后 3 个接近 0;
- **网络选择「丢弃」剩余 3 个特征**,因为同时存在它们,信号无法分离。

> 💡 **结论**:稠密 + 瓶颈 → 网络做硬选择,放弃次要特征。**没有叠加**。


## 4. 实验 2:稀疏数据让叠加发生

**关键改变**:`density = 0.1`(每个特征只有 10% 概率出现)。

> 📘 **直觉**:稀疏意味着「同时存在」的特征对很罕见。
> 网络可以**「赌」**:把多个特征塞进同一个方向,反正它们很少撞车。
> 偶尔撞车时,ReLU + 重要性权重让损失也能接受。这就给「叠加」开了大门。


In [ ]:
torch.manual_seed(1)
model_sparse = ToyModel(n_features, n_hidden)
losses = train(model_sparse, n_features, density=0.1,
               importance=importance, n_steps=8_000)
print(f'最终 loss: {losses[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_W(model_sparse.W, importance,
       title='Sparse (density=0.1): superposition emerges', ax=axes[0])
plot_W_norms(model_sparse.W,
             title='Feature representation magnitude', ax=axes[1])
plt.tight_layout()
plt.show()


**观察**:
- **5 个向量都获得了非零长度**——所有特征都被表示;
- 但它们**不再正交**——必然有某种几何排列让相邻特征「错开」;
- 这就是**叠加**:网络在 2 维里塞下了 5 个特征。

> 💡 **代价**:任意两个非正交特征同时激活时,输出会有干扰。
> 但因为稀疏,这种干扰罕见,损失可接受。**这是个工程上的权衡,不是免费午餐。**


## 5. 实验 3:几何相变(三角形 → 五边形)

为了让相变看得最清楚,我们做一个**特征等重要**(`I_i = 1`)的实验,
扫描密度从 1.0 到 0.01,观察 W 的几何形状如何变化。


In [ ]:
def run_uniform(n_features, n_hidden, density, n_steps=8_000, seed=0):
    torch.manual_seed(seed)
    model = ToyModel(n_features, n_hidden)
    importance = torch.ones(n_features)
    train(model, n_features, density=density,
          importance=importance, n_steps=n_steps)
    return model


densities = [1.0, 0.3, 0.1, 0.05, 0.02, 0.01]
n_features = 5
n_hidden = 2
importance_uni = torch.ones(n_features)

fig, axes = plt.subplots(1, len(densities), figsize=(3.2 * len(densities), 4))
for ax, d in zip(axes, densities):
    m = run_uniform(n_features, n_hidden, d, n_steps=6_000)
    plot_W(m.W, importance_uni, title=f'density = {d}', ax=ax)
plt.suptitle(f'5 features → 2 hidden: shape vs density (uniform importance)', y=1.02)
plt.tight_layout()
plt.show()


**观察相变**:
- `density=1.0`:仍然只学 2 个特征(2D 中两个正交方向);
- `density=0.3`:看到三角形雏形(3 个特征排成等边三角);
- `density=0.1`:更接近 5 边形;
- `density ≤ 0.05`:**清晰的正五边形**,5 个特征均匀分布在单位圆上;

这就是**几何相变**——sparsity 推动网络在「学多少特征」之间跳变。
论文里把这种最优排列称为「**对称构造**」(symmetric configurations),
n 个特征塞进 2 维的最优解就是正 n 边形。


### 5.1 高维的相变:更多特征塞进 2 维

把特征数量扩展到 8、12、20,看网络能否依然「塞下」。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, nf in zip(axes, [8, 12, 20]):
    m = run_uniform(n_features=nf, n_hidden=2, density=0.02, n_steps=10_000)
    plot_W(m.W, torch.ones(nf), title=f'{nf} features → 2 hidden', ax=ax)
plt.suptitle('Increasingly sparse → arbitrarily many features in 2D', y=1.02)
plt.tight_layout()
plt.show()


**惊人的事**:**只要数据足够稀疏,网络可以把任意多的特征塞进 2 维**。
这正是大模型为什么能在有限隐藏维度里编码海量概念的根本原因。

> ⚠️ **代价**:特征之间的角度变小,相互干扰增加,**需要更多训练数据 / 更稀疏的真实分布**才能稳定。


## 6. 加分:用 Sparse Autoencoder 拆开叠加的特征

到这里我们看到一个矛盾:
- **网络内部**:特征叠加在一起,神经元多义,人类无法直接读懂;
- **实际任务**:我们想知道「这个神经元代表什么」。

Anthropic 的解决方案就是 **Sparse Autoencoder (SAE)**——一个**辅助网络**,
把模型的隐藏激活 `h` 重新编码成一个**更宽、更稀疏**的表示 `z`,
让 `z` 的每个维度对应**一个**原始特征。这是 *Scaling Monosemanticity*(2024) 的核心方法。

### 6.1 SAE 的结构

```text
  h ∈ R^d  →  Encoder  →  z ∈ R^k(k 远大于 d, 但要求稀疏)  →  Decoder  →  ĥ
```

- 输入维度 `d` = 玩具模型的隐藏维度(2);
- 输出维度 `k` = 想拆出多少个「字典项」(应 ≥ n,即原始特征数);
- 损失 = **重构损失** + λ × **稀疏惩罚**(L1 让 z 大多数维度为 0);

如果一切顺利,SAE 学到的「字典」(decoder 的列向量)应该恰好对应原始特征方向。


In [ ]:
class SparseAutoencoder(nn.Module):
    """h → z (sparse, k-dim) → h_hat"""
    def __init__(self, d_hidden: int, k_dict: int):
        super().__init__()
        self.encoder = nn.Linear(d_hidden, k_dict, bias=True)
        self.decoder = nn.Linear(k_dict, d_hidden, bias=False)
        # Initialize decoder columns to be unit-norm (standard practice)
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)

    def forward(self, h):
        z = F.relu(self.encoder(h))
        h_hat = self.decoder(z)
        return z, h_hat


def train_sae(sae, get_activations, n_steps=8_000, batch_size=2048,
              lr=1e-3, l1=1e-2):
    sae = sae.to(DEVICE)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    for step in range(n_steps):
        h = get_activations(batch_size).detach()
        z, h_hat = sae(h)
        recon = ((h - h_hat) ** 2).mean()
        sparsity = z.abs().mean()
        loss = recon + l1 * sparsity
        opt.zero_grad()
        loss.backward()
        opt.step()
        # Renormalize decoder columns each step (standard SAE trick)
        with torch.no_grad():
            sae.decoder.weight.data = F.normalize(sae.decoder.weight.data, dim=0)
    return sae


### 6.2 训练玩具模型 → 收集激活 → 训练 SAE

我们用 `n=5, d=2, density=0.05` 训一个有叠加的玩具模型,
然后让 SAE 试图从它的隐藏激活里恢复出 5 个独立特征。


In [ ]:
# 1) 训练有叠加的玩具模型
torch.manual_seed(42)
n_features, n_hidden = 5, 2
toy = ToyModel(n_features, n_hidden).to(DEVICE)
importance_uni = torch.ones(n_features).to(DEVICE)
train(toy, n_features, density=0.05, importance=importance_uni, n_steps=8_000)

print('Toy model 训练完成,W 形状:', toy.W.shape)
plot_W(toy.W, importance_uni, title='Toy model superposition (n=5, d=2)')
plt.show()


In [ ]:
# 2) 训练 SAE 从激活里还原
def get_acts(batch_size):
    x = generate_batch(batch_size, n_features, density=0.05)
    return x @ toy.W   # h = W x

torch.manual_seed(123)
sae = SparseAutoencoder(d_hidden=2, k_dict=8)   # 字典宽度 8 > 5
train_sae(sae, get_acts, n_steps=8_000, l1=3e-2)

# Decoder columns = 'features' the SAE believes the activations are made of
D = sae.decoder.weight.detach()    # shape: (d_hidden, k_dict)
print('SAE 字典形状:', D.shape)


### 6.3 比较 SAE 字典 vs 真实特征方向

如果 SAE 真的拆开了叠加,**它的字典向量应该和玩具模型的 W 行向量方向一致(允许重新排列)**。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Plot toy model's true features
plot_W(toy.W, importance_uni,
       title='Truth: toy model feature directions (W rows)', ax=axes[0])

# Plot SAE dictionary (decoder columns) — only those with nonzero usage
W_sae = D.T  # (k_dict, 2)
# Filter out unused dictionary atoms (those with very small mean activation)
def get_atom_usage(sae, n_samples=4096):
    h = get_acts(n_samples).detach()
    with torch.no_grad():
        z = F.relu(sae.encoder(h))
    return z.mean(dim=0).cpu().numpy()
usage = get_atom_usage(sae)
active = usage > 0.01
print(f'活跃的字典项 ({active.sum()}/{len(usage)}):', np.where(active)[0].tolist())

# We'll color all atoms by usage
W_active = W_sae[active]
plot_W(W_active, torch.tensor(usage[active]),
       title=f'SAE dictionary (active atoms only)', ax=axes[1])

plt.tight_layout()
plt.show()


**观察**:
- 玩具模型的 5 个特征方向应该排成一个五边形;
- SAE 的 8 个字典项里**只有 5 个真正活跃**——这些活跃的方向应该和玩具模型的 W 行重合(可能旋转 / 镜像);
- **未活跃的 3 个字典项「死掉」了**——SAE 自动判断「不需要那么多」。

这就是 *Scaling Monosemanticity* 的玩具版:把神经网络的多义激活拆解为可解释、稀疏、单义的字典项。

> ⚠️ **现实差距**:在 Claude 这个尺度上做 SAE,字典宽度要到几百万,
> 训练成本是模型本身的若干倍,而且每个字典项是否真的「单义」仍要靠人类标注验证。
> 但**核心思想就是这一节看到的——不变。**


## 7. 总结与进一步阅读

### 你学到的核心概念

| 概念 | 一句话记忆 |
| --- | --- |
| **特征**(feature) | 数据里有意义的一个维度 / 概念 |
| **多义性**(polysemanticity) | 一个神经元同时承载多个特征 |
| **叠加**(superposition) | 网络在 d 维空间里塞下 n > d 个特征,只要它们足够稀疏 |
| **几何相变** | 特征数量 n 增加 → 表征自动变成正 n 边形 |
| **Sparse Autoencoder** | 把多义激活拆解为单义字典项,可解释性的关键工具 |

### 为什么这件事对 Claude 这种大模型重要

- Claude 的隐藏状态可能在每个位置只有几千维,但要表示**几百万个概念**;
- 唯一的解释:**真实输入是稀疏的**(任何一段文字只触发一小部分概念),所以叠加可行;
- 想读懂 Claude 内部 → 必须用 SAE 类工具拆开多义激活 → 这是 Anthropic *Scaling Monosemanticity* 项目的全部意义。

### 进一步阅读

按从易到难推荐:

1. **Anthropic 原论文 [*Toy Models of Superposition*](https://transformer-circuits.pub/2022/toy_model/index.html)**(2022) — 比这个 notebook 深 10 倍,但语言依然友好;
2. **[*Towards Monosemanticity*](https://transformer-circuits.pub/2023/monosemantic-features/index.html)**(2023) — SAE 在小语言模型上的首次成功;
3. **[*Scaling Monosemanticity*](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html)**(2024) — SAE 应用到 Claude 3 Sonnet,极其壮观的可视化;
4. **[Anthropic Circuits](https://transformer-circuits.pub/)** — 整个可解释性团队的论文索引;
5. **Neel Nanda 的 [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens)** — 一个让你能在真实小模型上做这类实验的工具库。

### 一个开放思考题

> 我们的玩具模型只有 2 个隐藏维度、5 个特征,SAE 要拆出 5 项就要 8 个字典槽。
> 在真实 LLM 上,这个比例(超完备倍数)通常是 **8× 到 64×**——也就是你想拆 1000 个概念,
> 字典要 8000 到 64000 项。**为什么需要这么超完备?**
> 思考完再去读 *Scaling Monosemanticity*,你会发现 Anthropic 给出了非常精彩的实证答案。
